# Worked examples

Six real ledger lines, computed step by step from `data/`, without the pipeline. The numbers are the ones in
`## Worked examples` of [`docs/design.md`](../docs/design.md). Each example ends with **R:**, the result.

- This notebook uses pandas; the pipeline uses SQL. Two separate computations that give the same rows are the check.
- R1, Q1, D1, ... are the rules, open questions and data issues in `docs/`.

## Setup

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data")

# Read as text, like the exploration notebook
ledger = pd.read_csv(DATA / "ledger.csv", dtype=str)
entities = pd.read_csv(DATA / "entities.csv", dtype=str)
accounts = pd.read_csv(DATA / "accounts.csv", dtype=str)
cost_centers = pd.read_csv(DATA / "cost_centers.csv", dtype=str)
pools = pd.read_csv(DATA / "pools.csv", dtype=str)
headcount_games = pd.read_csv(DATA / "headcount_games.csv", dtype=str)
headcount_employees = pd.read_csv(DATA / "headcount_employees.csv", dtype=str)
game_mapping = pd.read_csv(DATA / "game_mapping.csv", dtype=str)

# The only cleaning: trim the one untrimmed name found in 01-explore, check 2 (D2).
# The pipeline trims every name column used to match tables (Q8).
game_mapping["game_group"] = game_mapping["game_group"].str.strip()

ledger["amount_usd"] = ledger["amount_usd"].astype(float)
headcount_games["fte"] = headcount_games["fte"].astype(float)

In [2]:
# One ledger line with its entity group, account and cost center
def line(line_id):
    row = ledger[ledger["line_id"] == line_id]
    row = row.merge(entities, on="entity").merge(accounts, on="account").merge(cost_centers, on="cost_center", how="left")
    return row.iloc[0]


# The mapping row of a game or bucket that applies to a month (R10)
def mapping(name, month):
    first_day = month + "-01"
    rows = game_mapping[
        (game_mapping["game_group"] == name)
        & (game_mapping["start_date"] <= first_day)
        & (game_mapping["end_date"] >= first_day)
    ]
    assert len(rows) <= 1, f"two mapping rows for {name} in {month} (Q3)"
    return rows

### 1. Direct, single game (R4, R6, R7)

In [3]:
l = line("L00105")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["amount_usd"])

# The cost center has a game, so the whole line goes to it (share 1)
m = mapping(l["game"], l["month"])
pd.DataFrame([{
    "game": l["game"], "share": 1, "amount": round(l["amount_usd"], 2),
    "bearing_group": l["entity_group"],  # direct: bearing group = booked group (R6)
    "side": m["side"].iloc[0],
}])

2026-01 LGB Salaries 1001 Star Racer 103255.1


,game,share,amount,bearing_group,side
0,Star Racer,1,103255.1,BR,BR Games


**R:** Star Racer gets 103,255.10, bearing group BR, side BR Games. The mapping row only matches after the trim
(D2).

### 2. Direct, pool (R5)

In [4]:
l = line("L00054")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["pool_id"], l["amount_usd"])

# The pool's games, and their FTE that month
pool_games = pools[pools["pool_id"] == l["pool_id"]]
fte = pool_games.merge(headcount_games[headcount_games["month"] == l["month"]], on="game", how="left")
print(fte[["game", "fte"]])

# Only games with FTE > 0 share the line
fte = fte[fte["fte"] > 0].copy()
total_fte = fte["fte"].sum()
fte["share"] = fte["fte"].map("{:g}".format) + " / " + f"{total_fte:g}"
fte["amount"] = (l["amount_usd"] * fte["fte"] / total_fte).round(2)
fte["bearing_group"] = l["entity_group"]
fte["side"] = [mapping(g, l["month"])["side"].iloc[0] for g in fte["game"]]
fte = fte.sort_values("game")
print("total:", round(fte["amount"].sum(), 2))
fte[["game", "share", "amount", "bearing_group", "side"]]

2026-01 LGB Salaries 2010 New Games Prototyping NEW_SHARED 21972.14
          game   fte
0  Puzzle Pals  11.5
1   Moon Miner   NaN
2   Pixel Farm  14.5
3   Rune Quest   9.0
4    Tiny Chef   6.0
total: 21972.14


,game,share,amount,bearing_group,side
2,Pixel Farm,14.5 / 41,7770.63,BR,US Games
0,Puzzle Pals,11.5 / 41,6162.92,BR,BR Games
3,Rune Quest,9 / 41,4823.15,BR,US Games
4,Tiny Chef,6 / 41,3215.44,BR,BR Games


**R:**
- Moon Miner has no FTE in January, so it gets nothing; the other four share 41 FTE.
- Puzzle Pals and Tiny Chef go to BR Games, Pixel Farm and Rune Quest to US Games. All four keep bearing group BR.
- The rounded pieces add up to the line, 21,972.14.

### 3. Indirect (R8, R9, Q4)

In [5]:
l = line("L00444")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["amount_usd"])

# Bucket of each employee that month: first word of the category + entity group, in lower case
employees = headcount_employees[headcount_employees["month"] == l["month"]].merge(entities, on="entity", how="left")
employees["entity_group"] = employees["entity_group"].fillna("unknown")  # Q1 note: live_unknown, new_unknown
employees["bucket"] = employees["category"].str.split().str[0].str.lower() + "_" + employees["entity_group"].str.lower()

buckets = employees.groupby("bucket").size().reset_index(name="employees")
total = buckets["employees"].sum()
buckets["share"] = buckets["employees"].astype(str) + " / " + str(total)
buckets["amount"] = (l["amount_usd"] * buckets["employees"] / total).round(2)
buckets["bearing_group"] = [mapping(b, l["month"])["bearing_group"].iloc[0] for b in buckets["bucket"]]
buckets["side"] = [mapping(b, l["month"])["side"].iloc[0] for b in buckets["bucket"]]
print("total:", round(buckets["amount"].sum(), 2))
buckets[["bucket", "share", "amount", "bearing_group", "side"]]

2026-02 LGB Salaries 3301 Finance 21346.02
total: 21346.02


,bucket,share,amount,bearing_group,side
0,live_br,71 / 123,12321.69,BR,BR Games
1,new_br,25 / 123,4338.62,BR,BR Games
2,new_fi,8 / 123,1388.36,FI,US Games
3,new_others,5 / 123,867.72,Others,US Games
4,new_us,14 / 123,2429.63,US,US Games


**R:** 123 employees in February, in five buckets. The bearing group and side come from each bucket's mapping row.
The rounded pieces add up to the line, 21,346.02.

### 4. By entity (R7)

In [6]:
rows = []
for line_id in ["L00334", "L00336"]:
    l = line(line_id)
    side = mapping(l["game"], l["month"])["side"].iloc[0]
    print(line_id, l["game"], "mapping side:", side)
    # By entity: BR Games when booked by LGB, US Games for any other entity
    if side == "By entity":
        side = "BR Games" if l["entity"] == "LGB" else "US Games"
    rows.append({"line": line_id, "month": l["month"], "entity": l["entity"], "account": l["account_name"],
                 "cost_center": l["cost_center"], "amount": l["amount_usd"],
                 "bearing_group": l["entity_group"], "side": side})
pd.DataFrame(rows)

L00334 Legacy Arcade mapping side: By entity
L00336 Legacy Arcade mapping side: By entity


,line,month,entity,account,cost_center,amount,bearing_group,side
0,L00334,2026-01,LGB,Salaries,1004,24203.53,BR,BR Games
1,L00336,2026-01,LGI,Salaries,1004,12825.36,US,US Games


**R:** Same game and month: booked by LGB it goes to BR Games, booked by LGI to US Games.

### 5. Mapping dates (R10)

In [7]:
rows = []
for line_id in ["L00114", "L00494"]:
    l = line(line_id)
    side = mapping(l["game"], l["month"])["side"].iloc[0]
    rows.append({"line": line_id, "month": l["month"], "entity": l["entity"], "account": l["account_name"],
                 "cost_center": l["cost_center"], "amount": l["amount_usd"],
                 "bearing_group": l["entity_group"], "side": side})
print(game_mapping[game_mapping["game_group"] == "Puzzle Pals"][["game_group", "side", "start_date", "end_date"]])
pd.DataFrame(rows)

    game_group      side  start_date    end_date
4  Puzzle Pals  BR Games  2025-01-01  2026-02-28
5  Puzzle Pals  US Games  2026-03-01  9999-12-31


,line,month,entity,account,cost_center,amount,bearing_group,side
0,L00114,2026-02,LGB,Salaries,2001,31929.39,BR,BR Games
1,L00494,2026-03,LGB,Salaries,2001,32256.22,BR,US Games


**R:** Puzzle Pals is BR Games in February and US Games in March. The bearing group stays BR: only the side changes.

### 6. Unallocated (Q1, D1, D3)

In [8]:
# Whole line: the cost center is not in cost_centers.csv
l = line("L00169")
print(l["line_id"], l["month"], l["entity"], l["account_name"], l["cost_center"], l["amount_usd"])
print("department:", l["department"])

L00169 2026-02 LGB Professional Services 3999 17017.98
department: nan


**R:** No department, so the line has no path: one row, share 1, `unallocated`, `unknown_cost_center` (D1).

In [9]:
# One piece: a bucket with no mapping row
l = line("L00234")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["amount_usd"])

employees = headcount_employees[headcount_employees["month"] == l["month"]].merge(entities, on="entity", how="left")
employees["entity_group"] = employees["entity_group"].fillna("unknown")  # Q1 note: live_unknown, new_unknown
employees["bucket"] = employees["category"].str.split().str[0].str.lower() + "_" + employees["entity_group"].str.lower()

buckets = employees.groupby("bucket").size().reset_index(name="employees")
total = buckets["employees"].sum()
buckets["share"] = buckets["employees"].astype(str) + " / " + str(total)
buckets["amount"] = (l["amount_usd"] * buckets["employees"] / total).round(2)
buckets["has_mapping"] = [len(mapping(b, l["month"])) == 1 for b in buckets["bucket"]]
buckets["status"] = buckets["has_mapping"].map({True: "allocated", False: "unallocated"})
buckets["reason"] = buckets["has_mapping"].map({True: "", False: "no_mapping_row"})
# Bearing group and side only on allocated rows (Output, allocation.csv)
buckets["bearing_group"] = [mapping(b, l["month"])["bearing_group"].iloc[0] if ok else None
                            for b, ok in zip(buckets["bucket"], buckets["has_mapping"])]
buckets["side"] = [mapping(b, l["month"])["side"].iloc[0] if ok else None
                   for b, ok in zip(buckets["bucket"], buckets["has_mapping"])]
print("total:", round(buckets["amount"].sum(), 2))
buckets[["bucket", "share", "amount", "bearing_group", "side", "status", "reason"]]

2026-03 LGB Salaries 3301 Finance 24080.73


total: 24080.73


,bucket,share,amount,bearing_group,side,status,reason
0,live_br,70 / 125,13485.21,BR,BR Games,allocated,
1,live_fi,2 / 125,385.29,NaN,NaN,unallocated,no_mapping_row
2,new_br,26 / 125,5008.79,BR,BR Games,allocated,
3,new_fi,8 / 125,1541.17,FI,US Games,allocated,
4,new_others,5 / 125,963.23,Others,US Games,allocated,
5,new_us,14 / 125,2697.04,US,US Games,allocated,


**R:** live_fi (2 of 125 employees) has no mapping row, so its piece of 385.29 is `unallocated`, `no_mapping_row`
(D3). The other five pieces are allocated, and all six add up to the line, 24,080.73.